# 🎙️ Gagan AI - Custom Wake Word Training (OpenWakeWord)

Train a production-ready custom wake word model for **"Hey Gagan"** and **"Hey Gagan Rajput"**.

### 🛠️ Pre-resolved Fixes in this Notebook:
- ✅ **Python 3.13 Compatibility**: Bypasses `speexdsp-ns` & `piper-phonemize` wheel issues using `--no-deps`.
- ✅ **Authentic CMUDict**: Replaces the broken AWS S3 403 URL with the verified Hugging Face checkpoint.
- ✅ **Pre-downloaded ONNX Backbones**: Automatically fetches `embedding_model` and `melspectrogram`.
- ✅ **Torchaudio/SoundFile Patch**: Fixes `AttributeError: module torchaudio has no attribute info` in `torch_audiomentations`.
- ✅ **Lightweight Fast Features**: Uses the 184 MB background feature set instead of 17.3 GB download.
- ✅ **ONNXScript Pre-installed**: Prevents export failure at Step 3 line 905.
- ✅ **Auto-Download**: Automatically downloads `hey_gagan.onnx` to your computer when finished.


## 📦 Cell 1: Environment Setup & Clone Repositories
Installs dependencies, clones repos, and downloads TTS models and ONNX backbone feature extractors.

In [ ]:
# 1. Clone repos
!git clone -q https://github.com/dscripka/piper-sample-generator
!wget -q -nc -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!git clone -q https://github.com/dscripka/openwakeword

# 2. Install dependencies (with onnx, onnxscript, onnxruntime, soundfile)
!pip install -q onnx onnxscript onnxruntime soundfile
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 datasets
!pip install -q --no-deps -e ./openwakeword

# 3. Download ONNX backbone models into openwakeword resources
import os
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
!wget -q -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# 4. Download authentic CMUDict checkpoint (fixes AWS S3 403 error)
!wget -q -nc https://huggingface.co/TaterTotterson/DeepPhonemizer/resolve/main/en_us_cmudict_forward.pt -O en_us_cmudict_forward.pt
!cp en_us_cmudict_forward.pt ./openwakeword/
print('✅ Cell 1 Setup Complete!')


## 📥 Cell 2: Download Validation Background Features
Downloads the official precomputed openWakeWord negative audio features (~184 MB).

In [ ]:
!wget -q -nc https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print('✅ Validation background features downloaded!')


## 🛠️ Cell 3: Apply Torchaudio / SoundFile Patch
Ensures `torch_audiomentations` uses `soundfile` instead of the deprecated `torchaudio.info`.

In [ ]:
import os
# 1. Mock set_audio_backend in torchaudio
import torchaudio
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

# 2. Patch torch_audiomentations/utils/io.py on disk
patched = False
for root, dirs, files in os.walk('/usr/local/lib'):
    if 'torch_audiomentations' in root and 'io.py' in files:
        io_path = os.path.join(root, 'io.py')
        with open(io_path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            if 'set_audio_backend' in line and not line.strip().startswith('#'):
                new_lines.append('# ' + line)
            elif 'info = torchaudio.info' in line:
                new_lines.append('        import soundfile as _sf\n')
                new_lines.append('        _i = _sf.info(str(file_path))\n')
                new_lines.append('        return _i.frames, _i.samplerate\n')
            else:
                new_lines.append(line)
        with open(io_path, 'w') as f:
            f.writelines(new_lines)
        print(f'✅ Successfully patched: {io_path}')
        patched = True
if not patched:
    print('ℹ️ io.py already clean or not found.')


## ⚙️ Cell 4: Training Configuration for "Hey Gagan"
Defines the wake phrase (`hey gagan`, `hey gagan rajput`), sample size, and links background features.

In [ ]:
import yaml
config = yaml.load(open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader)
config['target_phrase'] = ['hey gagan', 'hey gagan rajput']
config['model_name'] = 'hey_gagan'
config['n_samples'] = 1000
config['n_samples_val'] = 250
config['steps'] = 2500
config['target_accuracy'] = 0.70
config['target_recall'] = 0.35
config['background_paths'] = []
config['rir_paths'] = []
os.makedirs('./mit_rirs', exist_ok=True)
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'validation_set_features.npy'}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print('✅ Saved my_model.yaml with target phrases: ["hey gagan", "hey gagan rajput"]')


## 🎙️ Step 1: Generate Synthetic Clips (~3-5 mins)
Uses Piper TTS to generate diverse positive and negative training clips.

In [ ]:
!python openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips


## 🔊 Step 2: Augment Clips & Extract Features (~2-3 mins)
Applies room reverberation, pitch shifting, and extracts openWakeWord embeddings.

In [ ]:
!python openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips


## 🧠 Step 3: Train Neural Network & Export ONNX (~1 min)
Trains the classifier and exports `hey_gagan.onnx` using `onnxscript`.

In [ ]:
!python openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model


## ⬇️ Step 4: Download `hey_gagan.onnx` to Computer
Saves the trained model to your PC.

In [ ]:
import glob, os
from google.colab import files
models = glob.glob('/content/**/hey_gagan.onnx', recursive=True)
if models:
    print(f'🎉 SUCCESS! Model found at: {models[0]} ({os.path.getsize(models[0])} bytes)')
    files.download(models[0])
    print('⬇️ Check your browser downloads for hey_gagan.onnx!')
else:
    print('⚠️ Model not found in /content, checking output folder:')
    !ls -la /content/my_custom_model/
